# Домашня робота №7: Time Series та JSONB для ML

**NYC Taxi + синтетичний event-log · PostgreSQL · Google Colab**

Запуск: **Runtime → Restart session and run all**. Комірки йдуть у порядку виконання. Відтворюваний fallback: якщо Parquet недоступний, генерується 60 днів × 5 зон.

SQL записаний багаторядково. Висновки з `EXPLAIN` слід уточнити після реального запуску.

## 1. Підготовка середовища та PostgreSQL

In [1]:
%pip install -q pixeltable-pgserver "psycopg[binary]" sqlalchemy pandas kagglehub

import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:467: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


### NYC Taxi: завантаження однієї місячної партиції

Офіційне джерело: [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). Використовуємо перші 500 000 рядків та фільтруємо аномальні значення. Якщо файл не завантажиться, спрацює fallback.

In [2]:
import pyarrow.parquet as pq
from pathlib import Path

URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet'
parquet_path = Path('/content/yellow_tripdata_2024-03.parquet')
columns = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_distance',
    'fare_amount', 'passenger_count',
]
USE_REAL_TLC = False
try:
    if not parquet_path.exists():
        urlretrieve(URL, parquet_path)
    pf = pq.ParquetFile(parquet_path)
    frames = []
    n_rows = 0
    for batch in pf.iter_batches(batch_size=50_000, columns=columns):
        frame = batch.to_pandas()
        frames.append(frame)
        n_rows += len(frame)
        if n_rows >= 500_000:
            break
    df = pd.concat(frames, ignore_index=True).head(500_000)
    df = df[
        df['fare_amount'].between(0, 500, inclusive='neither')
        & df['trip_distance'].between(0, 100, inclusive='neither')
        & df['PULocationID'].notna()
        & df['tpep_pickup_datetime'].notna()
    ].copy()
    if len(df) == 0:
        raise ValueError('Після очищення не залишилося рядків')
    df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False, chunksize=10_000)
    USE_REAL_TLC = True
    print(f'Завантажено та очищено {len(df):,} рядків NYC Taxi')
    del df, frames
except Exception as exc:
    print('Не вдалося отримати NYC Taxi. Переходимо на stub:', str(exc)[:350])


Не вдалося отримати NYC Taxi. Переходимо на stub: name 'urlretrieve' is not defined


### Daily aggregation та typed-таблиця

`ts_day` є локальною датою NYC, тип `DATE`. У реальному сценарії використовуємо raw → daily; у fallback створюємо детерміновані значення.

In [3]:
real_statements = [
    """
DROP TABLE IF EXISTS t7_hw_taxi_daily;
""",
    """
CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;
""",
    """
ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
"""
]
stub_statements = [
    """
DROP TABLE IF EXISTS t7_hw_taxi_daily;
""",
    """
CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id  INTEGER NOT NULL,
    ts_day          DATE    NOT NULL,
    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);
""",
    """
INSERT INTO t7_hw_taxi_daily (
    pickup_zone_id,
    ts_day,
    trip_count,
    avg_fare,
    total_distance
)
SELECT
    z.zone_id,
    DATE '2024-03-31' - (59 - n) AS ts_day,
    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
FROM generate_series(0, 59) AS n
CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);
"""
]
statements = real_statements if USE_REAL_TLC else stub_statements
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print("Daily-таблиця створена:", "NYC Taxi" if USE_REAL_TLC else "stub")


Daily-таблиця створена: stub


In [4]:
query = """
SELECT pickup_zone_id, COUNT(*) AS n_days FROM t7_hw_taxi_daily GROUP BY pickup_zone_id ORDER BY pickup_zone_id LIMIT 10;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,n_days
0,1,60
1,2,60
2,3,60
3,4,60
4,5,60


In [5]:
query = """
SELECT COUNT(*) AS n_rows, MIN(ts_day) AS first_day, MAX(ts_day) AS last_day FROM t7_hw_taxi_daily;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,n_rows,first_day,last_day
0,300,2024-02-01,2024-03-31


### Часові типи: `TIMESTAMP` проти `TIMESTAMPTZ`

Raw pickup-час локальний для Нью-Йорка; для збереження моменту часу його можна інтерпретувати через `AT TIME ZONE`. Демонстраційна typed-таблиця створюється лише для реальних TLC-даних.

In [6]:
if USE_REAL_TLC:
    statements = [
        "DROP TABLE IF EXISTS t7_hw_taxi_trips_typed",
        """
        CREATE TABLE t7_hw_taxi_trips_typed AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            "DOLocationID"::INTEGER AS dropoff_zone_id,
            (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
            (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
            trip_distance::NUMERIC(10, 2) AS trip_distance,
            fare_amount::NUMERIC(10, 2) AS fare_amount,
            passenger_count::INTEGER AS passenger_count
        FROM nyc_taxi_raw
        WHERE fare_amount > 0 AND fare_amount < 500
          AND trip_distance > 0 AND trip_distance < 100
        """,
    ]
    with engine.begin() as conn:
        for statement in statements:
            conn.execute(text(statement))
    with engine.connect() as conn:
        display(pd.read_sql_query(text('SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type FROM t7_hw_taxi_trips_typed LIMIT 5'), conn))
else:
    print('Typed-trip demo пропущено: працюємо зі stub daily data')


Typed-trip demo пропущено: працюємо зі stub daily data


## 2. Time-series ML features

Всі віконні функції мають `PARTITION BY pickup_zone_id`. `RANGE 6 days PRECEDING` охоплює сім календарних днів включно з поточним. `LAG(..., 30)` на пропущених датах означає 30 попередніх **рядків**, не обов’язково календарних днів; нижче додамо calendar spine.

### Lag 1/7/30

In [7]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-01,550,NaN,NaN,None
1,1,2024-02-02,587,550.0,NaN,None
2,1,2024-02-03,624,587.0,NaN,None
3,1,2024-02-04,661,624.0,NaN,None
4,1,2024-02-05,698,661.0,NaN,None
5,1,2024-02-06,585,698.0,NaN,None
6,1,2024-02-07,622,585.0,NaN,None
7,1,2024-02-08,659,622.0,550.0,None
8,1,2024-02-09,696,659.0,587.0,None
9,1,2024-02-10,733,696.0,624.0,None


### Rolling 7-day average fare

In [8]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-01,13.2,13.200000,1
1,1,2024-02-02,13.6,13.400000,2
2,1,2024-02-03,14.0,13.600000,3
3,1,2024-02-04,14.4,13.800000,4
4,1,2024-02-05,13.2,13.680000,5
5,1,2024-02-06,13.6,13.666667,6
6,1,2024-02-07,14.0,13.714286,7
7,1,2024-02-08,14.4,13.885714,7
8,1,2024-02-09,13.2,13.828571,7
9,1,2024-02-10,13.6,13.771429,7


### Rolling 30-day fare volatility

In [9]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-01,13.2,NaN,1
1,1,2024-02-02,13.6,0.282843,2
2,1,2024-02-03,14.0,0.400000,3
3,1,2024-02-04,14.4,0.516398,4
4,1,2024-02-05,13.2,0.521536,5
5,1,2024-02-06,13.6,0.467618,6
6,1,2024-02-07,14.0,0.445079,7
7,1,2024-02-08,14.4,0.478091,8
8,1,2024-02-09,13.2,0.489898,9
9,1,2024-02-10,13.6,0.463801,10


### Expanding cumulative trips

In [10]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-01,550,550,550.000000
1,1,2024-02-02,587,1137,568.500000
2,1,2024-02-03,624,1761,587.000000
3,1,2024-02-04,661,2422,605.500000
4,1,2024-02-05,698,3120,624.000000
5,1,2024-02-06,585,3705,617.500000
6,1,2024-02-07,622,4327,618.142857
7,1,2024-02-08,659,4986,623.250000
8,1,2024-02-09,696,5682,631.333333
9,1,2024-02-10,733,6415,641.500000


### Calendar spine: заповнення пропущених дат

На реальних даних деякі зони можуть не мати поїздок у певні дні. Створимо окрему щільну таблицю календарних дат, щоб `LAG(..., 7)` відповідав 7 календарним дням. Нульові `trip_count` та `total_distance` допустимі, а `avg_fare` без поїздок лишається `NULL`.

In [11]:
query = """
CREATE OR REPLACE VIEW v_t7_hw_taxi_calendar AS
WITH bounds AS (
    SELECT MIN(ts_day) AS min_day, MAX(ts_day) AS max_day
    FROM t7_hw_taxi_daily
),
zones AS (
    SELECT DISTINCT pickup_zone_id FROM t7_hw_taxi_daily
),
calendar AS (
    SELECT gs.day::DATE AS ts_day
    FROM bounds
    CROSS JOIN LATERAL generate_series(bounds.min_day, bounds.max_day, INTERVAL '1 day') AS gs(day)
)
SELECT z.pickup_zone_id, c.ts_day,
       COALESCE(d.trip_count, 0) AS trip_count,
       d.avg_fare,
       COALESCE(d.total_distance, 0)::NUMERIC(12, 2) AS total_distance
FROM zones z
CROSS JOIN calendar c
LEFT JOIN t7_hw_taxi_daily d
  ON d.pickup_zone_id = z.pickup_zone_id AND d.ts_day = c.ts_day;
"""
with engine.begin() as conn:
    conn.execute(text(query))
print("SQL виконано")


SQL виконано


### Combined feature view

Об’єднуємо lag, rolling та expanding у `v_t7_hw_taxi_features`. Для календарної точності використовуємо щільний calendar spine як джерело; решта логіки з шаблону збережена.

In [12]:
query = """
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM v_t7_hw_taxi_calendar
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
"""
with engine.begin() as conn:
    conn.execute(text(query))
print("SQL виконано")


SQL виконано


In [13]:
query = """
SELECT * FROM v_t7_hw_taxi_features ORDER BY pickup_zone_id, ts_day LIMIT 20;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,760,14.0,2370.0,873,801,550,13.714286,0.442563,22105,713.064516
1,1,2024-03-03,797,14.4,2395.0,760,838,587,13.885714,0.454049,22902,715.687500
2,1,2024-03-04,834,13.2,2420.0,797,725,624,13.828571,0.466831,23736,719.272727
3,1,2024-03-05,871,13.6,2445.0,834,762,661,13.771429,0.454049,24607,723.735294
4,1,2024-03-06,908,14.0,2470.0,871,799,698,13.714286,0.442563,25515,729.000000
5,1,2024-03-07,795,14.4,2495.0,908,836,585,13.885714,0.454049,26310,730.833333
6,1,2024-03-08,832,13.2,2520.0,795,873,622,13.828571,0.466831,27142,733.567568
7,1,2024-03-09,869,13.6,2545.0,832,760,659,13.771429,0.454049,28011,737.131579
8,1,2024-03-10,906,14.0,2570.0,869,797,696,13.714286,0.442563,28917,741.461538
9,1,2024-03-11,943,14.4,2595.0,906,834,733,13.885714,0.454049,29860,746.500000


## 3. JSONB event-log: 50 000 синтетичних подій

П’ять event types: `page_view`, `click`, `add_to_cart`, `checkout`, `recommendation_shown`. Поле `payload` має різні структури; `event_ts` зберігається як `TIMESTAMPTZ`.

In [14]:
statements = [
    """
DROP TABLE IF EXISTS t7_hw_events;
""",
    """
CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);
""",
    """
INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;
""",
    """
ANALYZE t7_hw_events;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f"Виконано {len(statements)} SQL-команд")


Виконано 4 SQL-команд


In [15]:
query = """
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### JSONB containment `@>`

In [16]:
query = """
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### JSONB key existence `?` та `?|`

Оператори перевіряють ключі на верхньому рівні JSONB.

In [17]:
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [18]:
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONPath: `jsonb_path_query` і `jsonb_path_exists`

In [19]:
query = """
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [20]:
query = """
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


## 4. Індексація JSONB та EXPLAIN ANALYZE

Спочатку вимірюємо без індексів, потім створюємо GIN `jsonb_path_ops` для `@>` та expression B-tree для `payload ->> 'device'`. PostgreSQL може обрати Seq Scan навіть після створення індексу, якщо це дешевше.

### Без GIN: baseline

In [21]:
statements = [
    """
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
""",
    """
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
""",
    """
DROP INDEX IF EXISTS idx_t7_events_device_btree;
""",
    """
ANALYZE t7_hw_events;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f"Виконано {len(statements)} SQL-команд")


Виконано 4 SQL-команд


In [22]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Aggregate  (cost=1759.84..1759.85 rows=1 width=8) (actual time=70.628..70.629 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3936 width=0) (actual time=0.016..68.261 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.194 ms
Execution Time: 70.655 ms


### Створення GIN `jsonb_path_ops`

In [23]:
statements = [
    """
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
""",
    """
ANALYZE t7_hw_events;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f"Виконано {len(statements)} SQL-команд")


Виконано 2 SQL-команд


In [24]:
query = """
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,gin_index_size
0,1144 kB


### Containment після GIN: два сценарії

In [25]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Aggregate  (cost=1099.05..1099.06 rows=1 width=8) (actual time=18.727..18.729 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=35.90..1090.03 rows=3609 width=0) (actual time=2.530..18.457 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..34.99 rows=3609 width=0) (actual time=0.363..0.363 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.288 ms
Execution Time: 18.764 ms


In [26]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Aggregate  (cost=1067.20..1067.21 rows=1 width=8) (actual time=15.057..15.059 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=25.37..1061.22 rows=2390 width=0) (actual time=0.470..13.809 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.78 rows=2390 width=0) (actual time=0.313..0.314 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.164 ms
Execution Time: 15.093 ms


In [27]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=34.425..34.426 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.019..30.661 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.078 ms
Execution Time: 34.453 ms


### Створення expression B-tree index та повторний EXPLAIN

In [28]:
statements = [
    """
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
""",
    """
ANALYZE t7_hw_events;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f"Виконано {len(statements)} SQL-команд")


Виконано 2 SQL-команд


In [29]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Aggregate  (cost=1245.70..1245.71 rows=1 width=8) (actual time=12.132..12.133 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.79..1224.57 rows=8452 width=0) (actual time=0.595..8.248 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..95.68 rows=8452 width=0) (actual time=0.450..0.450 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.241 ms
Execution Time: 12.166 ms


## 5. JSONB проти нормалізованої checkout-таблиці

In [30]:
statements = [
    """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
""",
    """
CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);
""",
    """
INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';
""",
    """
CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);
""",
    """
ANALYZE t7_hw_checkouts_norm;
"""
]
with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))
print(f"Виконано {len(statements)} SQL-команд")


Виконано 5 SQL-команд


In [31]:
query = """
SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;
"""
with engine.connect() as conn:
    result = pd.read_sql_query(text(query), conn)
display(result.head(20))


,n_checkouts
0,10000


### JSONB aggregation: EXPLAIN

In [32]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Limit  (cost=1953.81..1953.82 rows=3 width=72) (actual time=20.856..20.858 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1953.81..1967.63 rows=5528 width=72) (actual time=20.854..20.856 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.44..1882.36 rows=5528 width=72) (actual time=20.825..20.846 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.92 rows=9968 width=132) (actual time=0.015..12.443 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=14 read=2
Planning Time: 0.246 ms
Execution Time: 20.916

### Normalized aggregation: EXPLAIN

In [33]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
"""
with engine.connect() as conn:
    plan = conn.execute(text(query)).scalars().all()
print("\n".join(plan))


Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=5.243..5.245 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=5.242..5.243 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=5.231..5.233 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.010..0.979 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.488 ms
Execution Time: 5.284 ms
